# KBase Module Registration Testing

This notebook tests the KBase Catalog client for programmatic module registration.

**Purpose:**
- Test CatalogClient initialization and authentication
- Check module registration status
- View module information and build history
- Optionally trigger new registrations and promotions

**Environment:** Uses AppDev by default (can be changed to prod, ci, next)

## Step 1: Initialize Utilities

Load the NotebookUtil and CatalogClient. This cell initializes the utility class
which provides access to the KBase token and configuration.

In [ ]:
%run util.py

# Configuration for module registration
MODULE_NAME = "KBDatalakeApps"
GIT_URL = "https://github.com/kbaseapps/KBDatalakeApps"
ENVIRONMENT = "appdev"  # Options: appdev, prod, ci, next

# Get the catalog client with authentication
catalog = util.get_catalog_client(environment=ENVIRONMENT)

print(f"Module: {MODULE_NAME}")
print(f"Git URL: {GIT_URL}")
print(f"Environment: {ENVIRONMENT}")
print(f"Catalog URL: {catalog.url}")
print("\nCatalogClient initialized successfully!")

# Save configuration for later cells
util.save('registration_config', {
    'module_name': MODULE_NAME,
    'git_url': GIT_URL,
    'environment': ENVIRONMENT
})

## Step 2: Check if Module is Registered

Verify whether the module is already registered in the Catalog.
This is a quick check before attempting any registration operations.

In [ ]:
%run util.py

# Load configuration
config = util.load('registration_config')
catalog = util.get_catalog_client(environment=config['environment'])

# Check registration status
try:
    result = catalog.is_registered(module_name=config['module_name'])
    is_registered = result.get('is_registered', False) if result else False
    print(f"Module '{config['module_name']}' is registered: {is_registered}")
except CatalogError as e:
    print(f"Error checking registration: {e}")
    is_registered = False

# Save result
util.save('is_registered', {'is_registered': is_registered})

## Step 3: Get Module Information

Retrieve detailed information about the module including:
- Module description and language
- Owners list
- Version information (dev, beta, release)
- Git repository details

In [ ]:
%run util.py
import json

# Load configuration
config = util.load('registration_config')
catalog = util.get_catalog_client(environment=config['environment'])

# Get module info
try:
    module_info = catalog.get_module_info(module_name=config['module_name'])
    
    if module_info:
        print(f"=== Module: {module_info.get('module_name', 'N/A')} ===")
        print(f"Description: {module_info.get('description', 'N/A')}")
        print(f"Language: {module_info.get('language', 'N/A')}")
        print(f"Owners: {', '.join(module_info.get('owners', []))}")
        print(f"Git URL: {module_info.get('git_url', 'N/A')}")
        
        # Version info
        print("\n--- Version Information ---")
        for version_type in ['dev', 'beta', 'release']:
            version_info = module_info.get(version_type)
            if version_info:
                print(f"\n{version_type.upper()}:")
                print(f"  Git commit: {version_info.get('git_commit_hash', 'N/A')[:12]}...")
                print(f"  Version: {version_info.get('version', 'N/A')}")
                print(f"  Timestamp: {version_info.get('timestamp', 'N/A')}")
            else:
                print(f"\n{version_type.upper()}: Not available")
        
        # Save for later use
        util.save('module_info', module_info)
    else:
        print("No module information returned")
        
except CatalogError as e:
    print(f"Error getting module info: {e}")
    print("\nThis may mean the module is not yet registered.")

## Step 4: Get Module State

Check the current state of the module including:
- Registration status
- Active/inactive status
- Release approval state

In [ ]:
%run util.py

# Load configuration
config = util.load('registration_config')
catalog = util.get_catalog_client(environment=config['environment'])

# Get module state
try:
    state = catalog.get_module_state(module_name=config['module_name'])
    
    if state:
        print(f"=== Module State: {config['module_name']} ===")
        for key, value in state.items():
            print(f"  {key}: {value}")
        
        util.save('module_state', state)
    else:
        print("No state information returned")
        
except CatalogError as e:
    print(f"Error getting module state: {e}")

## Step 5: List Recent Builds

View the build history for the module to see:
- Recent registration attempts
- Build success/failure status
- Git commits that were built

In [ ]:
%run util.py

# Load configuration
config = util.load('registration_config')
catalog = util.get_catalog_client(environment=config['environment'])

# List recent builds
try:
    builds = catalog.list_builds(module_name=config['module_name'], limit=5)
    
    if builds:
        print(f"=== Recent Builds for {config['module_name']} ===")
        for i, build in enumerate(builds, 1):
            print(f"\nBuild {i}:")
            print(f"  Registration ID: {build.get('registration_id', 'N/A')}")
            print(f"  Git Commit: {build.get('git_commit_hash', 'N/A')[:12]}...")
            print(f"  Status: {build.get('registration', 'N/A')}")
            print(f"  Timestamp: {build.get('timestamp', 'N/A')}")
        
        util.save('recent_builds', builds)
    else:
        print("No builds found for this module")
        
except CatalogError as e:
    print(f"Error listing builds: {e}")

## Step 6: Register Module (Optional)

**WARNING:** This cell will trigger a new module registration/build.
Only run if you want to update the dev version in the Catalog.

This will:
1. Submit the git repository for registration
2. Trigger a Docker image build
3. Update the 'dev' version in the Catalog

In [ ]:
%run util.py

# Load configuration
config = util.load('registration_config')
catalog = util.get_catalog_client(environment=config['environment'])

# Safety check - set to True to actually register
CONFIRM_REGISTRATION = False  # Change to True to proceed

if CONFIRM_REGISTRATION:
    print(f"Registering module from: {config['git_url']}")
    print("This may take several minutes...\n")
    
    try:
        result = catalog.register_repo(git_url=config['git_url'])
        print(f"Registration submitted!")
        print(f"Registration ID: {result.get('registration_id', 'N/A')}")
        
        util.save('registration_result', result)
        
    except CatalogError as e:
        print(f"Registration error: {e}")
else:
    print("Registration skipped. Set CONFIRM_REGISTRATION = True to proceed.")
    print(f"\nThis would register: {config['git_url']}")

## Step 7: Check Build Log (After Registration)

After triggering a registration, use this cell to check the build progress.
You'll need the registration_id from the previous step.

In [ ]:
%run util.py

# Load configuration and registration result
config = util.load('registration_config')
catalog = util.get_catalog_client(environment=config['environment'])

# Try to load registration result from previous cell
try:
    reg_result = util.load('registration_result')
    registration_id = reg_result.get('registration_id')
except:
    registration_id = None

# Or manually set the registration ID here:
# registration_id = "your_registration_id_here"

if registration_id:
    try:
        log = catalog.get_build_log(registration_id)
        print(f"=== Build Log for {registration_id} ===")
        print("\n" + "="*60 + "\n")
        # Print last 100 lines
        lines = log.strip().split('\n')
        for line in lines[-100:]:
            print(line)
        print("\n" + "="*60)
        print(f"Total lines: {len(lines)}")
        
    except CatalogError as e:
        print(f"Error getting build log: {e}")
else:
    print("No registration ID available.")
    print("Run the registration cell first or set registration_id manually.")

## Step 8: Push Dev to Beta (Optional)

**WARNING:** This will promote the current dev version to beta.
Only run after verifying the dev build is working correctly.

This makes the app available to testers in the beta environment.

In [ ]:
%run util.py

# Load configuration
config = util.load('registration_config')
catalog = util.get_catalog_client(environment=config['environment'])

# Safety check - set to True to actually push
CONFIRM_BETA_PUSH = False  # Change to True to proceed

if CONFIRM_BETA_PUSH:
    print(f"Pushing dev to beta for: {config['module_name']}")
    
    try:
        catalog.push_dev_to_beta(module_name=config['module_name'])
        print("Successfully pushed dev to beta!")
        
        # Get updated module info
        info = catalog.get_module_info(module_name=config['module_name'])
        if info and info.get('beta'):
            print(f"\nBeta version now at:")
            print(f"  Git commit: {info['beta'].get('git_commit_hash', 'N/A')[:12]}...")
            print(f"  Version: {info['beta'].get('version', 'N/A')}")
            
    except CatalogError as e:
        print(f"Error pushing to beta: {e}")
else:
    print("Beta push skipped. Set CONFIRM_BETA_PUSH = True to proceed.")
    print(f"\nThis would push dev to beta for: {config['module_name']}")

## Step 9: Request Release (Optional)

**WARNING:** This creates a release request that must be approved by KBase admins.
Only request release after thorough testing in beta.

This promotes the app to production visibility.

In [ ]:
%run util.py

# Load configuration
config = util.load('registration_config')
catalog = util.get_catalog_client(environment=config['environment'])

# Safety check - set to True to actually request release
CONFIRM_RELEASE_REQUEST = False  # Change to True to proceed

if CONFIRM_RELEASE_REQUEST:
    print(f"Requesting release for: {config['module_name']}")
    
    try:
        catalog.request_release(module_name=config['module_name'])
        print("Release request submitted!")
        print("A KBase admin will review and approve the release.")
            
    except CatalogError as e:
        print(f"Error requesting release: {e}")
else:
    print("Release request skipped. Set CONFIRM_RELEASE_REQUEST = True to proceed.")
    print(f"\nThis would request release for: {config['module_name']}")

## Step 10: List All Your Modules

View all modules owned by specific users.
Useful for seeing all modules you have access to manage.

In [ ]:
%run util.py

# Load configuration
config = util.load('registration_config')
catalog = util.get_catalog_client(environment=config['environment'])

# List modules by owner
OWNERS = ['chenry']  # Add your username(s) here

try:
    modules = catalog.list_basic_module_info(
        owners=OWNERS,
        include_released=True,
        include_unreleased=True
    )
    
    if modules:
        print(f"=== Modules owned by {OWNERS} ===")
        for i, mod in enumerate(modules, 1):
            print(f"\n{i}. {mod.get('module_name', 'N/A')}")
            print(f"   Language: {mod.get('language', 'N/A')}")
            print(f"   Git URL: {mod.get('git_url', 'N/A')}")
            
        util.save('my_modules', modules)
    else:
        print(f"No modules found for owners: {OWNERS}")
        
except CatalogError as e:
    print(f"Error listing modules: {e}")

## Summary

This notebook provides the following capabilities:

| Step | Action | Safe to Run |
|------|--------|-------------|
| 1-5 | Read-only operations (info, state, builds) | Yes |
| 6 | Register/update module | Requires confirmation |
| 7 | Check build logs | Yes |
| 8 | Push dev to beta | Requires confirmation |
| 9 | Request release | Requires confirmation |
| 10 | List your modules | Yes |

**Typical workflow:**
1. Run steps 1-5 to check current status
2. Make code changes and push to GitHub
3. Run step 6 to trigger a new build
4. Run step 7 to monitor build progress
5. Test the app in AppDev
6. Run step 8 to push to beta
7. Test the app in beta
8. Run step 9 to request production release